# CMS Medicare Part D — Data Exploration

**Prerequisite:** Run `00_build_database.ipynb` first to create `cms.db`.

This notebook explores the structure and contents of the Part D geography dataset
before committing to a specific research question. Goals:
- Understand the geographic breakdown (National vs State rows)
- Survey what drug categories and therapeutic areas are present
- Identify oncology drugs available for analysis
- Check data quality (suppression rates, missing values)

In [1]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt

db_path = r"C:\Users\palla\OneDrive\Documents\Coding Projects\CMS\database\cms.db"
conn = sqlite3.connect(db_path)

## Step 1: Dataset Overview

How many rows are at the National level vs State level?
How many unique drugs are in the dataset?

In [2]:
# Row counts by geographic level
geo_summary = pd.read_sql_query("""
    SELECT Prscrbr_Geo_Lvl, COUNT(*) AS rows, COUNT(DISTINCT Gnrc_Name) AS unique_drugs
    FROM part_d_geo
    GROUP BY Prscrbr_Geo_Lvl
""", conn)

print("Geographic level summary:")
print(geo_summary)

Geographic level summary:
  Prscrbr_Geo_Lvl    rows  unique_drugs
0        National    3607          1945
1           State  112329          1868


## Step 2: Top Drugs by Total Claims (National)

What are the most-claimed drugs nationally? This gives a sense of the therapeutic
areas represented and where the data is richest.

In [3]:
top_drugs = pd.read_sql_query("""
    SELECT Gnrc_Name, Brnd_Name, Tot_Clms, Tot_Benes, Tot_Drug_Cst
    FROM part_d_geo
    WHERE Prscrbr_Geo_Lvl = 'National'
    ORDER BY Tot_Clms DESC
    LIMIT 30
""", conn)

print("Top 30 drugs by total claims (National):")
top_drugs

Top 30 drugs by total claims (National):


,Gnrc_Name,Brnd_Name,Tot_Clms,Tot_Benes,Tot_Drug_Cst
0,Atorvastatin Calcium,Atorvastatin Calcium,68467282,16770381.0,1.000314e+09
1,Amlodipine Besylate,Amlodipine Besylate,47516290,11570255.0,4.070836e+08
2,Levothyroxine Sodium,Levothyroxine Sodium,41665462,8683487.0,6.533209e+08
3,Lisinopril,Lisinopril,36064986,8746660.0,3.317373e+08
4,Gabapentin,Gabapentin,35115029,7549556.0,6.556496e+08
5,Losartan Potassium,Losartan Potassium,33183284,8187304.0,4.874522e+08
6,Metoprolol Succinate,Metoprolol Succinate,29617871,7153109.0,5.874448e+08
7,Omeprazole,Omeprazole,28132175,7294693.0,4.221951e+08
8,Rosuvastatin Calcium,Rosuvastatin Calcium,25844520,6942272.0,6.041944e+08
9,Metformin Hcl,Metformin Hcl,23861633,5915242.0,2.474325e+08


## Step 3: Oncology Drug Search

Search for oral chemotherapy and targeted therapy agents likely to appear in Part D.
IV chemotherapy (5-FU, oxaliplatin) is billed under Medicare Part B, not Part D,
so it will not appear here. Oral agents taken at home are Part D covered.

In [4]:
# Search for oral oncology agents by generic name keywords
onc_keywords = [
    'capecitabine', 'temozolomide', 'imatinib', 'erlotinib', 'gefitinib',
    'lapatinib', 'sunitinib', 'sorafenib', 'everolimus', 'ibrutinib',
    'lenalidomide', 'pomalidomide', 'thalidomide', 'palbociclib',
    'ribociclib', 'abemaciclib', 'olaparib', 'rucaparib', 'niraparib',
    'venetoclax', 'osimertinib', 'alectinib', 'crizotinib'
]

keyword_filter = " OR ".join([f"LOWER(Gnrc_Name) LIKE '%{k}%'" for k in onc_keywords])

onc_drugs = pd.read_sql_query(f"""
    SELECT Gnrc_Name, Brnd_Name, Tot_Prscrbrs, Tot_Clms, Tot_Benes,
           ROUND(Tot_Drug_Cst, 0) AS Tot_Drug_Cst
    FROM part_d_geo
    WHERE Prscrbr_Geo_Lvl = 'National'
      AND ({keyword_filter})
    ORDER BY Tot_Clms DESC
""", conn)

print(f"Oncology drugs found: {len(onc_drugs)}")
onc_drugs

Oncology drugs found: 36


,Gnrc_Name,Brnd_Name,Tot_Prscrbrs,Tot_Clms,Tot_Benes,Tot_Drug_Cst
0,Lenalidomide,Revlimid,11315,216143,36944.0,3.858656e+09
1,Ibrutinib,Imbruvica,9115,151924,17085.0,2.370268e+09
2,Palbociclib,Ibrance,8585,128889,15999.0,2.019562e+09
3,Lenalidomide,Lenalidomide,8741,121722,20385.0,1.680454e+09
4,Imatinib Mesylate,Imatinib Mesylate,9949,115874,14601.0,2.108899e+08
5,Venetoclax,Venclexta,8944,89227,18750.0,8.139482e+08
6,Pomalidomide,Pomalyst,7108,78680,12722.0,1.708116e+09
7,Osimertinib Mesylate,Tagrisso,5422,70781,9373.0,1.230787e+09
8,Abemaciclib,Verzenio,5503,50457,8213.0,7.566754e+08
9,Everolimus,Everolimus,5471,36192,6716.0,2.405536e+08


## Step 4: State Coverage Check

For a geographic map to work, a drug needs state-level rows for most or all 50 states.
Check how many states have data for capecitabine specifically.

In [5]:
cape_states = pd.read_sql_query("""
    SELECT Prscrbr_Geo_Desc, Tot_Prscrbrs, Tot_Clms, Tot_Benes,
           ROUND(Tot_Drug_Cst, 0) AS Tot_Drug_Cst
    FROM part_d_geo
    WHERE Prscrbr_Geo_Lvl = 'State'
      AND LOWER(Gnrc_Name) LIKE '%capecitabine%'
    ORDER BY Tot_Clms DESC
""", conn)

print(f"States with capecitabine data: {len(cape_states)}")
cape_states

States with capecitabine data: 13


,Prscrbr_Geo_Desc,Tot_Prscrbrs,Tot_Clms,Tot_Benes,Tot_Drug_Cst
0,Georgia,7,33,None,3371.0
1,Wisconsin,6,28,None,1372.0
2,Minnesota,4,25,None,2730.0
3,California,10,24,None,1098.0
4,Washington,4,19,None,1427.0
5,Mississippi,3,17,None,1592.0
6,South Carolina,4,17,None,1059.0
7,Virginia,2,15,None,828.0
8,Kansas,3,14,None,2383.0
9,North Carolina,2,13,None,738.0


## Step 5: Suppression Rate

CMS suppresses values when fewer than 11 beneficiaries received a drug in a given area.
High suppression in a drug's state rows means the map will have gaps.

In [6]:
# Check suppression rate across all state-level rows
suppression = pd.read_sql_query("""
    SELECT
        COUNT(*) AS total_state_rows,
        SUM(CASE WHEN Tot_Clms IS NULL THEN 1 ELSE 0 END) AS suppressed_rows,
        ROUND(100.0 * SUM(CASE WHEN Tot_Clms IS NULL THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_suppressed
    FROM part_d_geo
    WHERE Prscrbr_Geo_Lvl = 'State'
""", conn)

print("State-level suppression summary:")
print(suppression)

State-level suppression summary:
   total_state_rows  suppressed_rows  pct_suppressed
0            112329                0             0.0
